<img src="https://raw.githubusercontent.com/Delarbol/semillero/master/Images-DS/alt_banner.png" width=100%>
<h1 style="margin:0">Inteligencia Artificial Aplicada a la Economía II</h1>
<h2 style="margin:.35em 0 0 0">Sistemas de recomendación: del producto relevante al valor y la exposición</h2>
<p style="margin:.8em 0 0 0"><strong>Material guiado · Taller 5 · duración aproximada: 3 horas</strong></p>
</div>

> El caso de tienda utiliza datos sintéticos con fines pedagógicos. La extensión de películas utiliza datos reales de MovieLens (ml-latest-small), con usuarios anonimizados.

# Antes de programar: ¿qué es un sistema de recomendación?

Un **sistema de recomendación** ayuda a seleccionar y ordenar artículos que podrían interesar a una persona cuando el catálogo es demasiado grande para revisarlo completo. Un artículo —también llamado *ítem*— puede ser una película, un video, un libro o un producto. El sistema puede usar descripciones de artículos, interacciones anteriores, señales de otras personas y contexto. Una **interacción** puede ser una calificación, una compra, una reproducción o un clic; no todas expresan el mismo nivel de preferencia. Una recomendación suele ser una lista ordenada de candidatos, no una predicción causal de que el artículo aumentará el bienestar o las ventas. [1, 2]

**Ejemplo cotidiano.** Si a Ana le gustaron películas de aventura, una plataforma podría mostrarle más títulos de ese género. Si a personas con gustos similares a los de Ana les gustó una película que ella no ha visto, también podría aparecer en sus sugerencias. Las dos listas pueden diferir porque responden a señales distintas. [1, 2]

## Principales enfoques

| Enfoque | Señal que usa | Ejemplo sencillo | Límite que conviene revisar |
|---|---|---|---|
| **Popularidad (baseline)** | Frecuencia de compras, vistas o calificaciones en un periodo. | Mostrar películas populares a una persona sin historial. | Puede concentrar la exposición en artículos ya conocidos; no personaliza por gustos. |
| **Basado en contenido** | Atributos de los artículos y preferencias previas del usuario. | Recomendar otra película de aventura tras varias calificaciones altas en ese género. | Depende de descripciones útiles y puede repetir demasiado lo ya conocido. |
| **Filtrado colaborativo** | Patrones de interacción de muchos usuarios con muchos artículos. | Personas con historiales parecidos disfrutaron una película que Ana aún no vio. | Un usuario o artículo sin interacciones tiene poca señal: *arranque en frío*. |
| **Híbrido** | Combina contenido, colaboración y, si procede, contexto. | Usar géneros y valoraciones de otras personas para ordenar candidatos. | Combinar señales requiere definir objetivos y validar resultados. |
| **Contextual o secuencial** | Momento, dispositivo o secuencia reciente, además del historial. | Cambiar sugerencias según la sesión actual. | El contexto puede ser sensible y debe evaluarse con cuidado. |

Las categorías de contenido, colaboración e híbrido aparecen en [1] y [2]; [1] describe además el papel del contexto. **Popularidad** es una referencia simple que construiremos en el cuaderno. El análisis de canastas que veremos después encuentra artículos que aparecen juntos y puede alimentar recomendaciones complementarias; no es sinónimo de filtrado colaborativo.

### ¿Dónde aparecen estas ideas?

- **YouTube:** hay sugerencias en la página principal y junto al video que se está viendo, como “A continuación”. Es un ejemplo de selección y ordenamiento de videos a partir de señales de uso; no intentaremos reconstruir su sistema interno. [3]
- **Netflix:** la interfaz muestra filas de títulos y ordena los títulos dentro de ellas. Las calificaciones, el historial y los atributos de los títulos ilustran distintas fuentes de señal. Nuestro ejemplo MovieLens es un modelo de clase y **no** reproduce el algoritmo propietario de Netflix. [3]

**Preguntas para comenzar:** si una persona acaba de crear su cuenta, ¿cuáles enfoques pueden mostrar algo útil? Si aparece una película nueva sin calificaciones, ¿qué enfoque podría usar sus géneros? Formule una hipótesis y contraste sus respuestas después de ejecutar los ejemplos.

**Fuentes de esta introducción** (consultadas en septiembre de 2026):

1. NVIDIA, [“Recommendation System”](https://www.nvidia.com/en-us/glossary/recommendation-system/).
2. Intel, [“What Is a Recommendation System?”](https://www.intel.com/content/www/us/en/learn/recommendation-systems.html).
3. Insider One, [“Recommendation Systems: What You Need to Know”](https://insiderone.com/what-is-recommendation-system/), apartados sobre YouTube y Netflix.

# Qué aprenderemos

Una tienda puede recomendar productos de varias maneras. Aquí estudiaremos tres preguntas distintas:

1. **Contenido:** ¿qué artículos se parecen a los que interesaron a la persona?
2. **Colaboración:** ¿qué eligieron otras personas con patrones de compra parecidos?
3. **Transacciones:** ¿qué artículos suelen aparecer juntos en una misma canasta?

Después preguntaremos algo más exigente: **¿las recomendaciones funcionan para compras futuras, crean valor y distribuyen razonablemente la exposición?**

Al finalizar, cada estudiante o pareja entregará este mismo cuaderno con las celdas del Taller 5 y las respuestas de reflexión completas. Puede trabajar en grupos de máximo dos integrantes.

## Ruta de trabajo: 180 minutos

| Bloque | Tiempo sugerido | Producto |
|---|---:|---|
| Introducción: qué se recomienda y con qué señales | 15 min | Diferencias entre enfoques |
| 0. Datos, problema y baseline | 15 min | Datos auditados y referencia de popularidad |
| 1. Recomendación basada en contenido | 25 min | Perfil y lista de productos no vistos |
| 2. Filtrado colaborativo | 20 min | Lista basada en coincidencias entre usuarios |
| Extensión MovieLens | 20 min | Películas según géneros y calificaciones |
| 3. Análisis transaccional | 20 min | Regla de asociación interpretada |
| 4. Evaluación, valor y exposición | 20 min | Métricas y límites de la evidencia |
| 5. Taller 5 integrador | 45 min | Análisis, recomendaciones y reflexión |

En cada bloque hay un **ejemplo desarrollado**, un breve **su turno** para copiar y adaptar, y una pregunta sobre la intuición. El Taller 5 cambia los datos y reúne todo lo practicado.

# 0. Preparación y significado de los datos

Necesitamos `pandas`, `numpy` y `scikit-learn`. Si alguna importación falla, ejecute en otra celda:

```python
%pip install -q pandas numpy scikit-learn
```

Reinicie el kernel y vuelva a ejecutar el cuaderno. No se requiere conexión a una API ni archivos externos.

**Unidad de análisis:** cada fila registra un producto comprado en un pedido. Un pedido con tres productos ocupa tres filas. Por ello, `len(compras)` cuenta líneas de compra, no pedidos ni clientes.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

pd.set_option('display.max_colwidth', 100)
SEMILLA = 42
np.random.seed(SEMILLA)

## Datos del ejemplo desarrollado

Cada producto tiene una descripción, una categoría, un margen unitario hipotético y un grupo de proveedor. Un margen más alto no implica, por sí mismo, mayor bienestar del consumidor ni mayor incremento causal de ventas.

In [ ]:
productos = pd.DataFrame([
    ('p01', 'Café molido', 'bebidas', 'café molido tostado bebida desayuno', 4500, 'consolidado'),
    ('p02', 'Té verde', 'bebidas', 'té verde hojas bebida infusión', 3200, 'emergente'),
    ('p03', 'Avena integral', 'despensa', 'avena integral cereal desayuno', 2500, 'consolidado'),
    ('p04', 'Granola artesanal', 'despensa', 'granola avena cereal desayuno', 3800, 'emergente'),
    ('p05', 'Leche entera', 'lácteos', 'leche entera bebida desayuno', 1800, 'consolidado'),
    ('p06', 'Yogur natural', 'lácteos', 'yogur natural lácteo desayuno', 3000, 'emergente'),
    ('p07', 'Pasta seca', 'despensa', 'pasta seca comida almuerzo', 2000, 'consolidado'),
    ('p08', 'Salsa de tomate', 'despensa', 'salsa tomate para pasta almuerzo', 1900, 'consolidado'),
    ('p09', 'Chocolate oscuro', 'snacks', 'chocolate artesanal cacao merienda', 4300, 'emergente'),
    ('p10', 'Galletas de avena', 'snacks', 'galletas avena merienda desayuno', 2700, 'consolidado'),
], columns=['producto_id', 'nombre', 'categoria', 'descripcion', 'margen_unitario', 'grupo_proveedor'])

canastas_ejemplo = [
    ('u01', ['p01', 'p03', 'p05'], ['p03', 'p04']),
    ('u02', ['p01', 'p03', 'p04'], ['p01', 'p05']),
    ('u03', ['p01', 'p02', 'p09'], ['p01', 'p03']),
    ('u04', ['p03', 'p04', 'p06'], ['p05', 'p06']),
    ('u05', ['p07', 'p08'], ['p07', 'p08', 'p10']),
    ('u06', ['p07', 'p08'], ['p07', 'p10']),
    ('u07', ['p02', 'p09'], ['p02', 'p06']),
    ('u08', ['p05', 'p06', 'p10'], ['p03', 'p05', 'p06']),
    ('u09', ['p01', 'p09'], ['p01', 'p03', 'p10']),
]

def construir_compras(canastas, fecha_inicial='2026-08-01'):
    """Expande canastas ordenadas a líneas de compra reproducibles."""
    inicio = pd.Timestamp(fecha_inicial)
    filas = []
    for usuario, *pedidos in canastas:
        for numero_pedido, articulos in enumerate(pedidos, start=1):
            pedido_id = f'{usuario}_o{numero_pedido}'
            # Las rondas de pedidos comparten fecha: todo train precede a todo test.
            fecha = inicio + pd.Timedelta(days=7 * (numero_pedido - 1))
            for articulo in articulos:
                filas.append((pedido_id, usuario, fecha, articulo))
    return pd.DataFrame(filas, columns=['pedido_id', 'usuario_id', 'fecha', 'producto_id'])

compras = construir_compras(canastas_ejemplo)
display(productos.head())
display(compras.head(8))

## Auditoría mínima

Antes de calcular similitudes, verifique que los identificadores de productos existan, que no haya líneas repetidas dentro de un pedido y que las unidades estén claras.

In [ ]:
resumen = pd.Series({
    'líneas de compra': len(compras),
    'pedidos únicos': compras['pedido_id'].nunique(),
    'usuarios únicos': compras['usuario_id'].nunique(),
    'productos comprados': compras['producto_id'].nunique(),
    'líneas duplicadas en un pedido': compras.duplicated(['pedido_id', 'producto_id']).sum(),
    'productos sin ficha': (~compras['producto_id'].isin(productos['producto_id'])).sum(),
})
display(resumen.to_frame('valor'))
assert productos['producto_id'].is_unique
assert compras.duplicated(['pedido_id', 'producto_id']).sum() == 0
assert compras['producto_id'].isin(productos['producto_id']).all()

## Un baseline que debemos superar

El **baseline** recomienda los productos más populares en el historial disponible. Es fácil de implementar y puede ser sorprendentemente competitivo. Por simplicidad mediremos popularidad como **número de usuarios distintos que compraron el producto**, no como número total de líneas.

La función excluye los productos ya comprados por el usuario. Esta regla se usa en el cuaderno para recomendar **productos nuevos para esa persona**; en una tienda real algunas compras repetidas sí podrían ser valiosas.

In [ ]:
def productos_vistos(compras_hist, usuario):
    return set(compras_hist.loc[compras_hist['usuario_id'].eq(usuario), 'producto_id'])


def recomendar_populares(usuario, compras_hist, catalogo, k=3):
    vistos = productos_vistos(compras_hist, usuario)
    popularidad = compras_hist.groupby('producto_id')['usuario_id'].nunique()
    candidatos = catalogo[['producto_id', 'nombre']].copy()
    candidatos['puntaje'] = candidatos['producto_id'].map(popularidad).fillna(0)
    candidatos = candidatos.loc[~candidatos['producto_id'].isin(vistos)]
    return candidatos.sort_values(
        ['puntaje', 'producto_id'], ascending=[False, True]
    ).head(k).reset_index(drop=True)


display(recomendar_populares('u01', compras, productos, k=3))

### Su turno 0

Cambie el usuario por `u05`. ¿Por qué la lista puede resultar menos personalizada aunque sea razonable? Escriba su respuesta en la variable y conserve la celda ejecutada.

In [ ]:
display(recomendar_populares('u05', compras, productos, k=3))
respuesta_baseline = 'Escriba aquí una explicación de 2 a 3 frases.'
print(respuesta_baseline)

# 1. Filtrado basado en contenido

La recomendación **basada en contenido** compara los atributos de los productos. Si una persona compró avena, una descripción parecida podría llevar a granola o galletas de avena.

En nuestro ejemplo:

1. convertimos las descripciones en vectores **TF-IDF** (*term frequency–inverse document frequency*: frecuencia del término, ponderada por lo poco común que es en el catálogo);
2. promediamos los vectores de los artículos comprados para formar un perfil simple del usuario;
3. calculamos la **similitud coseno** entre perfil y candidatos;
4. excluimos productos ya vistos y ordenamos.

La similitud es un puntaje de cercanía bajo esta representación. **No** es una probabilidad de compra.

## Ejemplo desarrollado: construir la matriz de contenido

`fit_transform` aprende el vocabulario del catálogo y produce una fila por producto. Si en producción entran productos nuevos, debe definirse cómo actualizar esa representación.

In [ ]:
vectorizador = TfidfVectorizer(lowercase=True)
matriz_contenido = vectorizador.fit_transform(productos['descripcion'])
indice_producto = {p: i for i, p in enumerate(productos['producto_id'])}

print('Productos:', matriz_contenido.shape[0])
print('Términos del vocabulario:', matriz_contenido.shape[1])
display(pd.DataFrame(
    matriz_contenido.toarray(),
    index=productos['nombre'],
    columns=vectorizador.get_feature_names_out(),
).round(2).iloc[:5, :10])

## Ejemplo desarrollado: perfil y recomendaciones

Usaremos media simple de los artículos distintos que compró cada persona. Este perfil no considera recencia, cantidad, devoluciones ni preferencias declaradas.

In [ ]:
def recomendar_contenido(usuario, compras_hist, catalogo, matriz_texto, indice, k=3):
    vistos = productos_vistos(compras_hist, usuario)
    posiciones = [indice[p] for p in vistos if p in indice]
    if not posiciones:
        return pd.DataFrame(columns=['producto_id', 'nombre', 'puntaje'])

    perfil = np.asarray(matriz_texto[posiciones].mean(axis=0))
    similitudes = cosine_similarity(perfil, matriz_texto).ravel()
    candidatos = catalogo[['producto_id', 'nombre']].copy()
    candidatos['puntaje'] = similitudes
    candidatos = candidatos.loc[~candidatos['producto_id'].isin(vistos)]
    return candidatos.sort_values(
        ['puntaje', 'producto_id'], ascending=[False, True]
    ).head(k).reset_index(drop=True)


print('Historial u01:', sorted(productos_vistos(compras, 'u01')))
display(recomendar_contenido('u01', compras, productos, matriz_contenido, indice_producto, k=3))

### Su turno 1: copiar y adaptar

1. Use la función anterior con `u05` y `u07`.
2. Consulte las descripciones y explique una recomendación de cada lista.
3. ¿Qué ocurriría con una persona sin compras? ¿En qué se diferencia este caso de un producto nuevo con descripción?

Escriba la interpretación debajo del código. No basta con afirmar que “tienen similitud alta”: señale qué términos o atributos podrían explicar el resultado.

In [ ]:
for usuario in ['u05', 'u07']:
    print(f'\nUsuario {usuario}; vistos: {sorted(productos_vistos(compras, usuario))}')
    display(recomendar_contenido(usuario, compras, productos, matriz_contenido, indice_producto, k=3))

respuesta_contenido = """
Recomendación explicada para u05:
Recomendación explicada para u07:
Persona sin historial frente a producto nuevo:
"""
print(respuesta_contenido)

# 2. Filtrado colaborativo

El filtrado **colaborativo** usa patrones de interacción. No necesita leer la descripción del producto para detectar que ciertos productos aparecen en los historiales de usuarios parecidos.

Aquí usaremos colaboración **artículo–artículo**:

- construimos una matriz de usuarios por productos;
- cada celda vale 1 si el usuario compró al menos una vez el artículo, y 0 si no;
- comparamos columnas de productos mediante similitud coseno;
- puntuamos cada candidato por su similitud promedio con los artículos vistos por el usuario.

Los ceros significan “no observamos compra”, **no necesariamente “no le gusta”**. Este es un ejemplo de **retroalimentación implícita**.

## Ejemplo desarrollado: matriz usuario–producto

In [ ]:
def matriz_usuario_producto(compras_hist, catalogo):
    interacciones = compras_hist[['usuario_id', 'producto_id']].drop_duplicates().copy()
    interacciones['compro'] = 1
    matriz = interacciones.pivot_table(
        index='usuario_id', columns='producto_id', values='compro',
        aggfunc='max', fill_value=0,
    )
    return matriz.reindex(columns=catalogo['producto_id'], fill_value=0)


matriz_usuarios = matriz_usuario_producto(compras, productos)
display(matriz_usuarios)

In [ ]:
similitud_articulos = pd.DataFrame(
    cosine_similarity(matriz_usuarios.T),
    index=matriz_usuarios.columns,
    columns=matriz_usuarios.columns,
)
display(similitud_articulos.loc[['p03', 'p07'], ['p04', 'p05', 'p08', 'p10']].round(2))

## Ejemplo desarrollado: recomendar artículos no vistos

Para comparar candidatos, el puntaje suma sus similitudes con los artículos previamente comprados. No interpretaremos un puntaje de 0,8 como “80 % de probabilidad de comprar”.

In [ ]:
def recomendar_colaborativo(usuario, compras_hist, catalogo, k=3):
    matriz = matriz_usuario_producto(compras_hist, catalogo)
    if usuario not in matriz.index:
        return pd.DataFrame(columns=['producto_id', 'nombre', 'puntaje'])

    similitud = pd.DataFrame(
        cosine_similarity(matriz.T),
        index=matriz.columns, columns=matriz.columns,
    )
    vistos = set(matriz.columns[matriz.loc[usuario].eq(1)])
    candidatos = catalogo[['producto_id', 'nombre']].copy()
    candidatos['puntaje'] = candidatos['producto_id'].map(
        similitud.loc[:, list(vistos)].mean(axis=1)
    ).fillna(0)
    candidatos = candidatos.loc[~candidatos['producto_id'].isin(vistos)]
    return candidatos.sort_values(
        ['puntaje', 'producto_id'], ascending=[False, True]
    ).head(k).reset_index(drop=True)


display(recomendar_colaborativo('u01', compras, productos, k=3))

### Su turno 2: contenido frente a colaboración

Para `u05`, compare las dos listas. Señale una coincidencia o diferencia concreta y explique por qué cada método podría producirla. Si hay empate en el puntaje, el código desempata por identificador: ese orden no representa una preferencia aprendida.

¿Cuál método puede recomendar un producto nuevo con descripción pero sin compradores? ¿Cuál necesita interacciones de otros usuarios?

In [ ]:
usuario_comparacion = 'u05'
print('Basado en contenido:')
display(recomendar_contenido(
    usuario_comparacion, compras, productos, matriz_contenido, indice_producto, k=3
))
print('Colaborativo:')
display(recomendar_colaborativo(usuario_comparacion, compras, productos, k=3))

respuesta_comparacion = """
Coincidencia o diferencia observada:
Interpretación:
Producto nuevo y falta de interacciones:
"""
print(respuesta_comparacion)

## Extensión guiada: recomendar películas con MovieLens

Ahora aplicaremos **recomendación basada en contenido** a un conjunto de datos real. MovieLens `ml-latest-small` incluye una tabla de películas (`movieId`, título y géneros) y otra de calificaciones (`userId`, `movieId`, estrellas y marca de tiempo). Las calificaciones van de 0,5 a 5 estrellas. Los identificadores de usuarios están anonimizados. Los dos CSV necesarios están incorporados de forma comprimida en la siguiente celda, así que este archivo funciona sin descargas adicionales.

**Idea:** para un usuario concreto, consideraremos favoritas las películas calificadas con al menos 4 estrellas. Convertiremos los géneros en columnas indicadoras (1: pertenece; 0: no pertenece), promediaremos los géneros de sus favoritas y compararemos ese perfil con los géneros de otras películas mediante **similitud coseno**. Un valor más alto significa una dirección de atributos más parecida; **no** es una probabilidad de que el usuario vea o disfrute la película.

Para que la actividad sea rápida, trabajaremos con las 500 películas que recibieron más calificaciones en este conjunto. Esta elección favorece títulos populares: es una limitación del ejemplo, no un criterio universal de recomendación. Excluiremos **todas** las películas que el usuario ya calificó, incluso las que calificó mal. En este bloque exploraremos una lista, sin medir su desempeño predictivo. Una evaluación temporal exigiría construir catálogo y perfiles solo con datos disponibles antes de la fecha de prueba.

**Crédito del conjunto de datos:** GroupLens, `ml-latest-small`, generado el 26 de septiembre de 2018; consulte el `README.txt` distribuido con los archivos. Cita sugerida por sus autores: Harper, F. M. y Konstan, J. A. (2015), [“The MovieLens Datasets: History and Context”](https://doi.org/10.1145/2827872), *ACM Transactions on Interactive Intelligent Systems*, 5(4). Los datos se redistribuyen aquí para docencia bajo las condiciones indicadas en el `README.txt` de MovieLens.

In [ ]:
# Los CSV completos de MovieLens se incorporaron al cuaderno.
# Esta celda está plegada para que el texto comprimido no interrumpa la lectura.
import base64
import io
import zlib

def leer_csv_incorporado(cadena):
    datos = zlib.decompress(base64.b64decode(cadena))
    return pd.read_csv(io.BytesIO(datos))

peliculas_m = leer_csv_incorporado("""...""")
calificaciones_m = leer_csv_incorporado("""...
9BE=""")
print(f'{len(peliculas_m):,} películas; {len(calificaciones_m):,} calificaciones; '
      f'{calificaciones_m.userId.nunique()} usuarios')
display(peliculas_m.head(3))

### Ejemplo desarrollado: el perfil de géneros del usuario 1

1. Identificamos las 500 películas más calificadas para acotar el catálogo.
2. Para ese catálogo, transformamos géneros como `Adventure|Comedy` en columnas `Adventure` y `Comedy`.
3. Promediamos las filas de películas que el usuario 1 calificó con 4 estrellas o más: el resultado resume sus géneros frecuentes.
4. Calculamos similitud coseno con cada candidata; quitamos todo lo ya calificado y mostramos las cinco primeras.

**Entrada:** `usuario=1`, catálogo de 500 películas, calificaciones y `k=5`. **Salida:** una tabla con `movieId`, `title`, `genres` y `similitud`; no se espera que todas las películas tengan exactamente los mismos géneros.

In [ ]:
ids_catalogo_m = calificaciones_m['movieId'].value_counts().head(500).index
catalogo_m = (peliculas_m.loc[peliculas_m['movieId'].isin(ids_catalogo_m)]
              .sort_values('movieId').reset_index(drop=True))
indicadores_genero = catalogo_m['genres'].str.get_dummies(sep='|')
print('Columnas de géneros:', indicadores_genero.columns.tolist())

def recomendar_peliculas_por_genero(usuario, k=5):
    historial = calificaciones_m.loc[calificaciones_m['userId'].eq(usuario)]
    favoritos = set(historial.loc[historial['rating'].ge(4), 'movieId'])
    posiciones_favoritas = catalogo_m['movieId'].isin(favoritos)
    if not posiciones_favoritas.any():
        return pd.DataFrame(columns=['movieId', 'title', 'genres', 'similitud'])

    perfil = indicadores_genero.loc[posiciones_favoritas].mean(axis=0).to_numpy().reshape(1, -1)
    puntajes = cosine_similarity(perfil, indicadores_genero.to_numpy()).ravel()
    candidatos = catalogo_m.assign(similitud=puntajes)
    candidatos = candidatos.loc[~candidatos['movieId'].isin(historial['movieId'])]
    return (candidatos.sort_values(['similitud', 'movieId'], ascending=[False, True])
            [['movieId', 'title', 'genres', 'similitud']].head(k).reset_index(drop=True))

usuario_ejemplo = 1
favoritas_ejemplo = calificaciones_m.loc[
    calificaciones_m['userId'].eq(usuario_ejemplo) & calificaciones_m['rating'].ge(4),
    ['movieId', 'rating'],
].merge(peliculas_m[['movieId', 'title', 'genres']], on='movieId')
print(f'Usuario {usuario_ejemplo}: ejemplos de películas favoritas')
display(favoritas_ejemplo[['title', 'rating', 'genres']].head(5))
print(f'Cinco recomendaciones entre las {len(catalogo_m)} películas del catálogo:')
display(recomendar_peliculas_por_genero(usuario_ejemplo, k=5).round({'similitud': 3}))

### Su turno: aplicar e interpretar, sin atribuir resultados a Netflix

Copie la llamada anterior, cambie el identificador a `2` y después a `4`, y compare ambos top 5. Observe también al menos dos películas que cada persona calificó alto. Puede reutilizar la función desarrollada sin volver a escribirla.

Responda en el texto de la celda: (a) ¿qué género ayuda a explicar una sugerencia concreta para cada persona?, (b) ¿por qué una similitud alta no demuestra que la persona vaya a disfrutarla?, (c) ¿por qué excluimos todas las películas ya calificadas, incluso con puntuación baja?, y (d) ¿qué cambiaría para evaluar estos resultados con datos futuros en lugar de solo inspeccionar una lista?

In [ ]:
for usuario_otro in [2, 4]:
    print(f'\nUsuario {usuario_otro}: películas favoritas (muestra)')
    favoritos_otro = calificaciones_m.loc[
        calificaciones_m['userId'].eq(usuario_otro) & calificaciones_m['rating'].ge(4),
        ['movieId', 'rating'],
    ].merge(peliculas_m[['movieId', 'title', 'genres']], on='movieId')
    display(favoritos_otro[['title', 'rating', 'genres']].head(2))
    print('Top 5 sugerido:')
    display(recomendar_peliculas_por_genero(usuario_otro, k=5).round({'similitud': 3}))

respuesta_peliculas = """
(a) Géneros observados y sugerencia explicada para usuarios 2 y 4:
(b) Similitud frente a preferencia futura:
(c) Exclusión de toda película ya calificada:
(d) Evaluación de recomendaciones futuras:
"""
print(respuesta_peliculas)

# 3. Análisis transaccional: productos que aparecen juntos

La unidad ahora es el **pedido**, no el usuario. Preguntamos, por ejemplo: “Si un pedido contiene pasta, ¿cuán frecuentemente incluye también salsa?”.

Para una regla $A \Rightarrow B$:

- **Soporte conjunto** $P(A \cap B)$: proporción de pedidos que contienen ambos.
- **Confianza** $P(B\mid A)$: de los pedidos que contienen A, proporción que también contiene B.
- **Lift** $P(B\mid A)/P(B)$: compara la confianza con la frecuencia general de B. Un lift mayor que 1 indica coocurrencia por encima de la esperada bajo independencia.

Un lift alto **no demuestra causalidad** y puede ser inestable si el soporte es pequeño. Los resultados varían cuando cambia el periodo, el catálogo o la composición de compradores.

## Ejemplo desarrollado: regla pasta → salsa

Primero creamos una tabla con un pedido por fila y un indicador por producto. Después calculamos las tres medidas a partir de los conteos, con denominadores explícitos.

In [ ]:
def matriz_pedidos(compras_hist, catalogo):
    lineas = compras_hist[['pedido_id', 'producto_id']].drop_duplicates().copy()
    lineas['presente'] = 1
    matriz = lineas.pivot_table(
        index='pedido_id', columns='producto_id', values='presente',
        aggfunc='max', fill_value=0,
    )
    return matriz.reindex(columns=catalogo['producto_id'], fill_value=0)


pedidos_binarios = matriz_pedidos(compras, productos)
display(pedidos_binarios.head())

In [ ]:
def medir_regla(matriz_canastas, antecedente, consecuente):
    if antecedente == consecuente:
        raise ValueError('Antecedente y consecuente deben ser diferentes.')
    if antecedente not in matriz_canastas or consecuente not in matriz_canastas:
        raise KeyError('Producto ausente de la matriz de canastas.')

    a = matriz_canastas[antecedente].eq(1)
    b = matriz_canastas[consecuente].eq(1)
    n = len(matriz_canastas)
    n_a, n_b, n_ambos = int(a.sum()), int(b.sum()), int((a & b).sum())
    confianza = n_ambos / n_a if n_a else np.nan
    soporte_b = n_b / n if n else np.nan
    lift = confianza / soporte_b if soporte_b else np.nan
    return pd.Series({
        'n_pedidos': n, 'n_A': n_a, 'n_B': n_b, 'n_A_y_B': n_ambos,
        'soporte_conjunto': n_ambos / n if n else np.nan,
        'confianza': confianza, 'lift': lift,
    })


display(medir_regla(pedidos_binarios, 'p07', 'p08').to_frame('Pasta → Salsa'))

### Su turno 3: invertir e interpretar

1. Calcule la regla `p08 → p07` y compárela con `p07 → p08`.
2. ¿Por qué el soporte conjunto es igual en las dos direcciones, pero la confianza puede cambiar?
3. ¿Usaría esta regla para recomendar automáticamente a todas las personas? Considere la muestra pequeña y el contexto.

In [ ]:
display(medir_regla(pedidos_binarios, 'p08', 'p07').to_frame('Salsa → Pasta'))

respuesta_reglas = """
Comparación de soporte y confianza:
Decisión sobre uso automático y limitaciones:
"""
print(respuesta_reglas)

# 4. Evaluar utilidad, valor y exposición

Una recomendación puede parecer coherente al mirar todo el historial. Para evaluarla debemos simular una situación real: **recomendamos con el pasado y comparamos con el futuro**.

Separaremos por pedido completo:

- historial de cada usuario: todos sus pedidos salvo el último;
- prueba: último pedido de cada usuario;
- objetivo: artículos del pedido futuro que no aparecían en su historial.

El test no se usa para aprender popularidad, similitudes, reglas ni para ajustar el método. Este ejercicio es una **evaluación retrospectiva**; no observa qué habría comprado la persona sin recomendación.

In [ ]:
def separar_ultimo_pedido(compras_hist):
    pedidos = compras_hist[['usuario_id', 'pedido_id', 'fecha']].drop_duplicates()
    pedidos = pedidos.sort_values(['usuario_id', 'fecha', 'pedido_id'])
    ultimos = pedidos.groupby('usuario_id')['pedido_id'].last()
    ids_test = set(ultimos.to_numpy())
    train = compras_hist.loc[~compras_hist['pedido_id'].isin(ids_test)].copy()
    test = compras_hist.loc[compras_hist['pedido_id'].isin(ids_test)].copy()
    assert set(train['pedido_id']).isdisjoint(test['pedido_id'])
    assert train['usuario_id'].nunique() == test['usuario_id'].nunique()
    assert train['fecha'].max() < test['fecha'].min(), 'El corte global debe ser temporal.'
    return train, test


train, test = separar_ultimo_pedido(compras)
print('Pedidos históricos:', train['pedido_id'].nunique())
print('Pedidos futuros:', test['pedido_id'].nunique())

## Objetivo: novedades en el pedido futuro

Si el usuario ya compró café en el pasado y vuelve a comprarlo, no lo contaremos como acierto en esta evaluación de **descubrimiento**. Para estudiar recompra habría que diseñar otro objetivo y permitir recomendaciones repetidas.

In [ ]:
def novedades_por_usuario(train_hist, test_futuro):
    objetivos = {}
    for usuario, filas in test_futuro.groupby('usuario_id'):
        objetivos[usuario] = set(filas['producto_id']) - productos_vistos(train_hist, usuario)
    return objetivos


objetivos = novedades_por_usuario(train, test)
display(pd.DataFrame([
    {'usuario_id': usuario, 'productos_nuevos_en_test': sorted(items)}
    for usuario, items in objetivos.items()
]))

## Métricas de ranking a tres posiciones

Para cada usuario, tomando $K=3$:

- **Precision@3** = aciertos entre los tres recomendados / 3.
- **Recall@3** = aciertos / número de productos nuevos que efectivamente compró en test.
- **Hit@3** = 1 si hubo al menos un acierto, 0 si no.

Si no hay productos nuevos en test, Recall@3 no está definida para ese usuario: no la reemplazamos silenciosamente por cero. Promediamos Recall@3 solo donde el denominador existe. Precision@3 usa tres posiciones siempre que haya al menos tres candidatos.

In [ ]:
def evaluar_ranking(nombre, funcion_recomendar, train_hist, test_futuro, catalogo, k=3):
    objetivos_locales = novedades_por_usuario(train_hist, test_futuro)
    filas = []
    for usuario, reales in objetivos_locales.items():
        tabla = funcion_recomendar(usuario, train_hist, catalogo, k=k)
        recomendados = tabla['producto_id'].tolist()
        if len(recomendados) < k:
            raise ValueError(f'{nombre}: menos de {k} candidatos para {usuario}.')
        if set(recomendados) & productos_vistos(train_hist, usuario):
            raise ValueError(f'{nombre}: recomendó un producto ya visto a {usuario}.')
        aciertos = set(recomendados) & reales
        filas.append({
            'metodo': nombre, 'usuario_id': usuario,
            'recomendados': recomendados, 'nuevos_en_test': sorted(reales),
            'aciertos': sorted(aciertos), 'precision_at_k': len(aciertos) / k,
            'recall_at_k': len(aciertos) / len(reales) if reales else np.nan,
            'hit_at_k': int(bool(aciertos)),
        })
    return pd.DataFrame(filas)


comparacion = pd.concat([
    evaluar_ranking('Popularidad', recomendar_populares, train, test, productos),
    evaluar_ranking(
        'Contenido',
        lambda u, h, c, k: recomendar_contenido(u, h, c, matriz_contenido, indice_producto, k),
        train, test, productos,
    ),
    evaluar_ranking('Colaborativo', recomendar_colaborativo, train, test, productos),
], ignore_index=True)

display(comparacion.groupby('metodo', as_index=False)[
    ['precision_at_k', 'recall_at_k', 'hit_at_k']
].mean().round(3))

### Su turno 4: interpretar sin exagerar

1. ¿Cuál método alcanzó el mayor Hit@3 en **estos datos**? ¿Es una ventaja concluyente con nueve usuarios?
2. ¿Por qué no debemos llamar “tasa de conversión causada por el recomendador” a Hit@3?
3. ¿Qué pasaría si hubiéramos calculado la matriz colaborativa con todo el historial, incluido test?

In [ ]:
display(comparacion[['metodo', 'usuario_id', 'recomendados', 'nuevos_en_test', 'aciertos']])

respuesta_evaluacion = """
Mayor Hit@3 y cautela:
Por qué no es efecto causal:
Qué sería fuga de información:
"""
print(respuesta_evaluacion)

## Valor económico: un proxy, no incremento causal

Como ejercicio, sumaremos el margen unitario de los productos futuros que coinciden con el top 3. Lo llamaremos **margen coincidente**. No es utilidad incremental atribuible al recomendador: la compra se observó sin desplegar estas recomendaciones, y faltan costos de exposición, descuentos, sustitución y un grupo de control.

In [ ]:
mapa_margenes = productos.set_index('producto_id')['margen_unitario'].to_dict()

comparacion['margen_coincidente'] = comparacion['aciertos'].apply(
    lambda items: sum(mapa_margenes[p] for p in items)
)

display(comparacion.groupby('metodo', as_index=False).agg(
    hit_at_3=('hit_at_k', 'mean'),
    margen_coincidente_total=('margen_coincidente', 'sum'),
).round(3))

## Exposición: ¿quién aparece en las listas?

Los artículos populares tienden a recibir más visualizaciones. Un sistema que los favorece puede reforzar esa ventaja. Mediremos la fracción de **posiciones recomendadas** que corresponde a proveedores emergentes y la compararemos con su fracción del catálogo. Es un diagnóstico descriptivo, no una prueba automática de discriminación ni un objetivo universal de paridad.

También importan la posición en pantalla, la elegibilidad de productos y la diversidad de gustos; aquí contamos cada posición por igual.

In [ ]:
def tabla_exposicion(resultados, catalogo):
    grupos = catalogo.set_index('producto_id')['grupo_proveedor']
    filas = []
    for metodo, bloque in resultados.groupby('metodo'):
        expuestos = [p for lista in bloque['recomendados'] for p in lista]
        for grupo in sorted(catalogo['grupo_proveedor'].unique()):
            filas.append({
                'metodo': metodo,
                'grupo_proveedor': grupo,
                'fraccion_posiciones': sum(grupos[p] == grupo for p in expuestos) / len(expuestos),
                'fraccion_catalogo': catalogo['grupo_proveedor'].eq(grupo).mean(),
            })
    return pd.DataFrame(filas)


display(tabla_exposicion(comparacion, productos).round(3))

### Su turno 5: decisión responsable

¿Elegiría el método de mayor Hit@3, el de mayor margen coincidente o el de mejor distribución de exposición? Proponga un criterio de decisión que explicite la tensión. Después indique **qué experimento** permitiría estimar el incremento causal de valor al mostrar las recomendaciones, frente a no mostrarlas o mostrar el baseline.

In [ ]:
respuesta_decision = """
Criterio propuesto (incluya relevancia, valor y exposición):

Experimento y métrica económica principal:

Posible daño o sesgo que vigilaría:
"""
print(respuesta_decision)

# 5. Taller 5: personalización de una tienda de barrio

## Encargo

Una tienda quiere recomendar productos adicionales a clientes existentes. Cuenta con un catálogo y tres pedidos cronológicos por cliente. Deben construir una recomendación basada en contenido y otra colaborativa, contrastarlas con popularidad, analizar al menos una regla de canasta y elaborar una recomendación de negocio que considere relevancia, margen y exposición.

**Tiempo sugerido: 50 minutos.** Entreguen este cuaderno con código ejecutado y texto en los espacios de respuesta. Pueden copiar y adaptar las funciones del ejemplo desarrollado; no es necesario programar una librería nueva.

### Reglas del taller

- Use exclusivamente los pedidos 1 y 2 para construir recomendaciones y el pedido 3 como test.
- Recomiende tres **productos nuevos** por usuario; ningún producto histórico debe aparecer en el top 3.
- Calcule popularidad, similitudes y reglas de canasta **solo con train** cuando las use para decidir.
- No cambie puntajes, reglas ni hiperparámetros después de mirar el pedido 3. La comparación permite **proponer un piloto**, no declarar un ganador definitivo: para estimar desempeño futuro después de seleccionar la política haría falta un periodo nuevo.
- La comparación retrospectiva no constituye una estimación causal de ventas incrementales.

**Cómo completar las plantillas:** las variables que contienen `None` están pendientes. Sustituya `None` por una llamada o cálculo adaptado del ejemplo desarrollado. Las celdas posteriores muestran “Pendiente” hasta que termine la fase anterior; aun así, el cuaderno se puede ejecutar de principio a fin antes de completarlo.

## 5.1 Datos nuevos

Las categorías y descripciones son deliberadamente sencillas. Algunos productos provienen de proveedores emergentes. Los márgenes son hipotéticos y están expresados en pesos por unidad.

In [ ]:
integrantes = 'Nombre(s) y apellido(s): '
print(integrantes)

catalogo_taller = pd.DataFrame([
    ('t01', 'Arroz', 'despensa', 'arroz blanco grano comida almuerzo', 1800, 'consolidado'),
    ('t02', 'Lentejas', 'despensa', 'lentejas grano legumbre almuerzo', 2200, 'emergente'),
    ('t03', 'Aceite', 'despensa', 'aceite vegetal cocina almuerzo', 2400, 'consolidado'),
    ('t04', 'Harina', 'despensa', 'harina trigo cocina desayuno', 1600, 'consolidado'),
    ('t05', 'Huevos', 'frescos', 'huevos frescos desayuno cocina', 2000, 'consolidado'),
    ('t06', 'Café local', 'bebidas', 'café tostado bebida desayuno', 3900, 'emergente'),
    ('t07', 'Té de hierbas', 'bebidas', 'té hierbas bebida infusión', 2700, 'emergente'),
    ('t08', 'Galletas', 'snacks', 'galletas dulce merienda café', 1500, 'consolidado'),
    ('t09', 'Atún', 'conservas', 'atún lata proteína comida', 3000, 'consolidado'),
    ('t10', 'Pasta', 'despensa', 'pasta seca comida almuerzo', 1900, 'consolidado'),
    ('t11', 'Salsa casera', 'conservas', 'salsa tomate casera pasta', 3100, 'emergente'),
    ('t12', 'Jabón artesanal', 'aseo', 'jabón artesanal cuidado hogar', 2800, 'emergente'),
], columns=['producto_id', 'nombre', 'categoria', 'descripcion', 'margen_unitario', 'grupo_proveedor'])

canastas_taller = [
    ('v01', ['t01', 't02'], ['t03', 't01'], ['t05', 't02']),
    ('v02', ['t01', 't02'], ['t05', 't03'], ['t04', 't01']),
    ('v03', ['t10', 't11'], ['t09', 't10'], ['t11', 't03']),
    ('v04', ['t10', 't11'], ['t03', 't10'], ['t09', 't11']),
    ('v05', ['t06', 't08'], ['t07', 't08'], ['t06', 't12']),
    ('v06', ['t06', 't08'], ['t06', 't05'], ['t07', 't12']),
    ('v07', ['t07', 't08'], ['t07', 't04'], ['t06', 't08']),
    ('v08', ['t09', 't01'], ['t03', 't09'], ['t02', 't12']),
    ('v09', ['t12', 't05'], ['t04', 't12'], ['t03', 't05']),
    ('v10', ['t04', 't05'], ['t01', 't04'], ['t02', 't08']),
]

compras_taller = construir_compras(canastas_taller, fecha_inicial='2026-08-10')
display(catalogo_taller.head())
display(compras_taller.head())
print('Usuarios:', compras_taller['usuario_id'].nunique(),
      'pedidos:', compras_taller['pedido_id'].nunique(),
      'líneas:', len(compras_taller))

## Fase A. Auditar y separar (10 puntos)

Copie el patrón de la auditoría inicial. Compruebe número de pedidos, usuarios, productos sin ficha y duplicados. Después llame a `separar_ultimo_pedido` y verifique que no haya pedidos compartidos entre train y test.

**Pregunta A:** ¿por qué separar filas al azar, en vez de pedidos completos por fecha, contaminaría este ejercicio?

In [ ]:
auditoria_taller = pd.Series({
    'usuarios': compras_taller['usuario_id'].nunique(),
    'pedidos': None,                  # TODO: contar pedido_id distintos
    'duplicados_pedido_producto': None,  # TODO: adaptar duplicated del ejemplo
    'productos_sin_ficha': None,     # TODO: adaptar isin del ejemplo
})
display(auditoria_taller.to_frame('valor'))

train_taller, test_taller = separar_ultimo_pedido(compras_taller)
print('Pedidos train/test:', train_taller['pedido_id'].nunique(), test_taller['pedido_id'].nunique())
assert set(train_taller['pedido_id']).isdisjoint(test_taller['pedido_id'])

respuesta_A = 'Escriba aquí por qué la separación aleatoria por filas sería problemática.'
print(respuesta_A)

## Fase B. Baseline y contenido (20 puntos)

1. Calcule el baseline para `v05` usando únicamente `train_taller`.
2. Ajuste `TfidfVectorizer` a las descripciones de `catalogo_taller`.
3. Copie y adapte la llamada a `recomendar_contenido` para `v05`.
4. Verifique que el top 3 no contenga productos vistos en train.

**Pregunta B:** ¿qué recomendación se explica por los términos de la descripción? ¿Qué no se puede deducir de la similitud?

In [ ]:
vectorizador_taller = TfidfVectorizer(lowercase=True)
matriz_texto_taller = None  # TODO: ajustar el vectorizador al catálogo
indice_taller = None        # TODO: dict producto_id -> posición de fila

usuario_foco = 'v05'
baseline_foco = None    # TODO: adaptar recomendar_populares(...)
contenido_foco = None   # TODO: adaptar recomendar_contenido(...)

if baseline_foco is not None and contenido_foco is not None:
    print('Baseline:')
    display(baseline_foco)
    print('Contenido:')
    display(contenido_foco)
    assert len(contenido_foco) == 3
    assert not set(contenido_foco['producto_id']) & productos_vistos(train_taller, usuario_foco)
else:
    print('Fase B pendiente: complete el vectorizador, el índice y las dos recomendaciones.')

respuesta_B = """
Producto y términos que ayudan a entender la similitud:
Interpretación que NO puede hacerse a partir del puntaje:
"""
print(respuesta_B)

## Fase C. Filtrado colaborativo (15 puntos)

1. Construya la matriz de usuarios por productos usando `train_taller`.
2. Compare la lista colaborativa de `v05` con baseline y contenido.
3. Explique una diferencia y diga qué método podría trabajar con un artículo nuevo que solo tiene descripción.

**Pregunta C:** ¿por qué una celda cero de la matriz no significa que al cliente le desagrada el producto?

In [ ]:
matriz_colab_taller = None  # TODO: adaptar matriz_usuario_producto(...)
colaborativo_foco = None    # TODO: adaptar recomendar_colaborativo(...)

if matriz_colab_taller is not None and colaborativo_foco is not None:
    display(matriz_colab_taller.head())
    display(colaborativo_foco)
    assert len(colaborativo_foco) == 3
    assert not set(colaborativo_foco['producto_id']) & productos_vistos(train_taller, usuario_foco)
else:
    print('Fase C pendiente: construya la matriz y la recomendación colaborativa.')

respuesta_C = """
Comparación concreta de listas:
Producto nuevo con descripción:
Por qué cero no equivale a desagrado:
"""
print(respuesta_C)

## Fase D. Canastas de compra (15 puntos)

Construya la matriz de pedidos **solo con `train_taller`**. Calcule una regla con sentido comercial, por ejemplo pasta → salsa (`t10 → t11`), y la regla inversa. Reporte los conteos además de soporte, confianza y lift.

**Pregunta D:** ¿le parecen suficientes los pedidos que respaldan esa asociación? ¿Es válido concluir que comprar pasta causa la compra de salsa?

In [ ]:
matriz_pedidos_taller = None  # TODO: adaptar matriz_pedidos(...)
regla_AB = None  # TODO: medir t10 -> t11 con medir_regla(...)
regla_BA = None  # TODO: medir t11 -> t10 con medir_regla(...)

if regla_AB is not None and regla_BA is not None:
    display(pd.concat([regla_AB.rename('Pasta → Salsa'),
                       regla_BA.rename('Salsa → Pasta')], axis=1).round(3))
else:
    print('Fase D pendiente: calcule las dos direcciones con datos de train.')

respuesta_D = """
Interpretación con conteos y denominadores:
Confianza para tomar decisiones y límite causal:
"""
print(respuesta_D)

## Fase E. Evaluación temporal (20 puntos)

Copie el patrón de `evaluar_ranking` para comparar los tres métodos con `K=3`. La función de contenido recibe como argumentos adicionales la matriz de texto y el índice; adapte la llamada, no use `test_taller` para ajustar el modelo.

Reporte Precision@3, Recall@3 y Hit@3. Revise al menos un usuario para el que los métodos difieran.

**Pregunta E:** ¿cuál método funciona mejor bajo cada métrica en esta comparación exploratoria? ¿Por qué no alcanza esta muestra para afirmar que funcionará mejor en producción?

In [ ]:
# TODO: copie y adapte las tres llamadas a evaluar_ranking del ejemplo.
# Para Contenido, use una función lambda que pase matriz_texto_taller e indice_taller.
resultado_popularidad = None
resultado_contenido = None
resultado_colaborativo = None

if all(x is not None for x in [resultado_popularidad, resultado_contenido, resultado_colaborativo]):
    resultados_taller = pd.concat(
        [resultado_popularidad, resultado_contenido, resultado_colaborativo],
        ignore_index=True,
    )
    resumen_taller = resultados_taller.groupby('metodo', as_index=False)[
        ['precision_at_k', 'recall_at_k', 'hit_at_k']
    ].mean().round(3)
    display(resumen_taller)
    # TODO: seleccione un usuario con diferencias y muestre sus tres filas.
else:
    resultados_taller = None
    print('Fase E pendiente: evalúe los tres métodos sobre el test reservado.')

respuesta_E = """
Comparación de las tres métricas:
Ejemplo de un usuario y qué cambió:
Por qué el resultado no es concluyente:
"""
print(respuesta_E)

## Fase F. Valor, exposición y decisión (20 puntos)

1. Calcule el **margen coincidente** de los aciertos del test para cada método, siguiendo el ejemplo.
2. Construya `tabla_exposicion` para comparar la presencia de proveedores emergentes en las posiciones recomendadas frente a su proporción en el catálogo.
3. Elija una propuesta para una prueba piloto: puede ser un método o una combinación sencilla, pero indique qué resultados la justificarían y qué riesgo vigilará.

**Pregunta F:** ¿por qué el margen coincidente no equivale a valor incremental? Proponga un experimento A/B (asignación aleatoria entre dos políticas) con una métrica económica y una medida de exposición. No afirme que los datos sintéticos demuestran sesgo real.

In [ ]:
if resultados_taller is not None:
    mapa_margenes_taller = catalogo_taller.set_index('producto_id')['margen_unitario'].to_dict()
    # TODO: adapte el cálculo de margen_coincidente del ejemplo.
    # TODO: agrupe por método y muestre margen_coincidente_total y hit_at_3.
    exposicion_taller = None  # TODO: adapte tabla_exposicion(...)
    if exposicion_taller is not None:
        display(exposicion_taller.round(3))
else:
    print('Fase F pendiente: primero complete los resultados de la fase E.')

respuesta_F = """
Método o política que llevaría a un piloto y por qué:

Por qué margen coincidente no es impacto incremental:

Diseño del experimento A/B, métrica económica y medida de exposición:

Riesgo de sesgo o retroalimentación que vigilaría:
"""
print(respuesta_F)

## Rúbrica del Taller 5 (100 puntos)

| Componente | Puntos | Evidencia para obtener el puntaje completo |
|---|---:|---|
| A. Datos y separación | 10 | Auditoría correcta; train/test por pedido y fecha; explicación del riesgo de fuga. |
| B. Baseline y contenido | 20 | Top 3 sin artículos vistos; interpretación de términos y límite del puntaje. |
| C. Colaborativo | 15 | Matriz binaria correcta; comparación concreta y explicación de ceros y producto nuevo. |
| D. Canastas | 15 | Dos direcciones, conteos y medidas correctas; cautela por muestra y causalidad. |
| E. Evaluación | 20 | Métricas a K=3 con train exclusivo; lectura por usuario y límites de generalización. |
| F. Valor y exposición | 20 | Cálculo de proxy, comparación de exposición y piloto con experimento bien formulado. |
| **Total** | **100** | Código ejecutado, resultados interpretados y respuestas propias. |

Los cálculos sin interpretación, o las conclusiones que llamen causal al resultado retrospectivo, no obtienen el puntaje completo. Una conclusión que detecte ausencia de ventaja también puede ser excelente si está sustentada.

## Antes de entregar

- [ ] Ejecuté el cuaderno desde la primera celda hasta el final.
- [ ] Reemplacé todos los textos que dicen “Escriba aquí” por respuestas propias.
- [ ] No usé el último pedido para entrenar, construir similitudes o elegir reglas.
- [ ] Ningún top 3 incluye artículos ya vistos por el mismo usuario.
- [ ] Distinguí similitud, coocurrencia, acierto retrospectivo e impacto causal.
- [ ] Expliqué al menos una tensión entre relevancia, margen y exposición.

**Salida a entregar:** este archivo `.ipynb` con código y respuestas. Identifique en la primera celda de respuestas los nombres de quienes participaron.

In [ ]:
conclusion_final = """
En cinco o seis líneas, resuma la política recomendada, la evidencia que la apoya,
el principal límite de la evaluación y cómo la validaría en un piloto.
"""
print(conclusion_final)